# MVP-1 Ablation — A1 (Ranking) + A2 (Consistency) + A3 (DANN)

**Objetivo:** Descontaminar z_img de batch (study_part) sin matar señal biológica.

| ID | Pérdidas | Batch control | Qué valida |
|----|----------|---------------|------------|
| A0 | MAE(PDL) | ninguno | referencia (ya corrido en baseline) |
| A1 | MAE + ranking intra-línea | ninguno | robustez a PDL ruidoso entre donantes |
| A2 | MAE + ranking + consistency | ninguno | estabilidad del embedding |
| A3 | MAE + ranking + consistency + DANN(study_part) | gradient reversal | reducción de batch sin perder señal |

**Criterio de aceptación:**
- Batch incremental AUC (study_part) ≤ 0.65–0.70
- PDL bins AUC ≥ 0.85
- Fold peor: Spearman PDL ≥ 0.60
- Fusion-readiness: correlación parcial telómero/mtDNA controlando PDL y cell_line

---

## SECCIÓN 0 — CONFIG

In [1]:
import os

# ============================================================
# >>> AJUSTAR ESTAS 4 RUTAS <<<
# ============================================================
MANIFEST_PATH = "/Users/JCB/Documentos/Proyecto Integrador/data/manifests/manifest_mvp1_finetune_20260319_153838.csv"
CSV_CENTRAL   = "/Users/JCB/Documentos/Proyecto Integrador/data/Lifespan_Study_Data.csv"
IMAGE_ROOT    = "/Volumes/SanDisk SSD 1TB/Storage/Data/Cellular_Lifespan_Study_Brightfield_Images"
OUTPUT_DIR    = "/Users/JCB/Documentos/Proyecto Integrador/results/mvp1_ablation"

# ============================================================
# COLUMNAS DEL MANIFEST
# ============================================================
COL_IMG_PATH      = "img_path"
COL_PDL_NORM      = "pdl_norm"
COL_PDL_RAW       = "pdl"
COL_FOLD          = "fold"
COL_CELL_LINE     = "cell_line"
COL_STUDY_PART    = "study_part"
COL_MAGNIFICATION = "magnification"
COL_SAMPLE_ID     = "sample_id"
COL_TELOMERE      = "telomere_length"
COL_MTDNA         = "mtdna_cn"

# ============================================================
# HIPERPARÁMETROS BASE
# ============================================================
BATCH_SIZE    = 16
EPOCHS        = 40
PATIENCE      = 10
LR_HEAD       = 1e-3
WEIGHT_DECAY  = 1e-4
EMBEDDING_DIM = 256
IMG_SIZE      = 224
SEED          = 42

# ============================================================
# PESOS DE PÉRDIDAS (se varían por ablation)
# ============================================================
LAMBDA_PDL   = 1.0
LAMBDA_RANK  = 0.3
LAMBDA_CONS  = 0.2
LAMBDA_DANN  = 0.1     # empieza bajo, se escala con schedule
DANN_MAX     = 0.5     # lambda DANN máximo al final del entrenamiento

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Config cargada. Output → {OUTPUT_DIR}")

✅ Config cargada. Output → /Users/JCB/Documentos/Proyecto Integrador/results/mvp1_ablation


## SECCIÓN 1 — IMPORTS

In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import json
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Function
import torchvision.transforms as T
import torchvision.models as models

from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

from PIL import Image

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("🖥  Device: Apple MPS")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"🖥  Device: CUDA — {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("🖥  Device: CPU")

🖥  Device: Apple MPS


## SECCIÓN 2 — CARGAR MANIFEST (solo 10x)

In [3]:
df = pd.read_csv(MANIFEST_PATH)
print(f"Manifest cargado: {df.shape[0]} filas")

# Resolver rutas
sample_path = df[COL_IMG_PATH].iloc[0]
if not os.path.isabs(sample_path):
    df["_img_abs"] = df[COL_IMG_PATH].apply(lambda p: os.path.join(IMAGE_ROOT, p))
else:
    df["_img_abs"] = df[COL_IMG_PATH]

# Verificar existencia
exists_mask = df["_img_abs"].apply(os.path.exists)
df = df[exists_mask].reset_index(drop=True)

# Filtro 10x
n_antes = len(df)
df = df[df[COL_MAGNIFICATION].astype(str) == "10"].reset_index(drop=True)
print(f"🔬 Filtro 10x: {n_antes} → {len(df)} imágenes")

# Codificar study_part como entero para DANN
study_parts = sorted(df[COL_STUDY_PART].unique())
sp_to_idx = {sp: i for i, sp in enumerate(study_parts)}
df["_sp_idx"] = df[COL_STUDY_PART].map(sp_to_idx)
N_STUDY_PARTS = len(study_parts)

print(f"\n📊 Resumen:")
print(f"   Imágenes: {len(df)}")
print(f"   Cell lines: {sorted(df[COL_CELL_LINE].unique())}")
print(f"   Folds: {df[COL_FOLD].value_counts().sort_index().to_dict()}")
print(f"   Study parts: {study_parts} ({N_STUDY_PARTS} clases para DANN)")

Manifest cargado: 763 filas
🔬 Filtro 10x: 763 → 393 imágenes

📊 Resumen:
   Imágenes: 393
   Cell lines: ['hFB1', 'hFB11', 'hFB12', 'hFB13', 'hFB14', 'hFB2']
   Folds: {0.0: 127, 1.0: 217, 2.0: 49}
   Study parts: [1, 2, 3, 5] (4 clases para DANN)


/var/folders/fy/k8jygkfn6hd63_wrlzhk82c40000gp/T/ipykernel_56930/3385731534.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_img_abs"] = df[COL_IMG_PATH]
/var/folders/fy/k8jygkfn6hd63_wrlzhk82c40000gp/T/ipykernel_56930/3385731534.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_sp_idx"] = df[COL_STUDY_PART].map(sp_to_idx)


## SECCIÓN 3 — DATASET (con soporte para consistency loss)

In [4]:
class AblationDataset(Dataset):
    """
    Dataset que retorna:
      - img (transformada)
      - img_aug (segunda augmentation, para consistency)
      - pdl_norm
      - cell_line_idx (para ranking intra-línea)
      - study_part_idx (target de DANN)
    """

    def __init__(self, dataframe, transform, transform_aug=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.transform_aug = transform_aug or transform
        # Codificar cell_line
        self.cl_unique = sorted(self.df[COL_CELL_LINE].unique())
        self.cl_to_idx = {cl: i for i, cl in enumerate(self.cl_unique)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["_img_abs"])
        if img.mode != "RGB":
            img = img.convert("RGB")

        img1 = self.transform(img)
        img2 = self.transform_aug(img)

        pdl = torch.tensor(row[COL_PDL_NORM], dtype=torch.float32)
        cl_idx = torch.tensor(self.cl_to_idx[row[COL_CELL_LINE]], dtype=torch.long)
        sp_idx = torch.tensor(row["_sp_idx"], dtype=torch.long)

        return img1, img2, pdl, cl_idx, sp_idx


# Transforms
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Segunda augmentation (más agresiva, para consistency)
train_transform_aug = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.15, contrast=0.15),
    T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("✅ Dataset class y transforms definidos")

✅ Dataset class y transforms definidos


## SECCIÓN 4 — GRADIENT REVERSAL LAYER + MODELO

In [5]:
class GradientReversalFn(Function):
    """Gradient Reversal Layer (Ganin & Lempitsky, 2016)."""
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


class PDLEncoderDANN(nn.Module):
    """
    ResNet-34 (frozen) + embedding head + PDL head + DANN head.

    Produce:
      - pdl_hat: predicción de PDL normalizado
      - embedding: vector 256-dim
      - sp_logits: logits para study_part (DANN)
    """

    def __init__(self, embedding_dim=256, n_study_parts=4, freeze_backbone=True):
        super().__init__()

        backbone = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        backbone_out = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # Embedding head
        self.embed_head = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        # PDL regression head
        self.pdl_head = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

        # DANN head (adversarial: study_part classifier)
        self.dann_head = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_study_parts),
        )

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"   Params entrenables: {trainable:,} / {total:,}")

    def forward(self, x, dann_alpha=0.0):
        features = self.backbone(x)
        embedding = self.embed_head(features)
        pdl_hat = self.pdl_head(embedding).squeeze(-1)

        # Gradient reversal para DANN
        reversed_emb = GradientReversalFn.apply(embedding, dann_alpha)
        sp_logits = self.dann_head(reversed_emb)

        return pdl_hat, embedding, sp_logits


print("✅ GradientReversal + PDLEncoderDANN definidos")
_ = PDLEncoderDANN(EMBEDDING_DIM, N_STUDY_PARTS, freeze_backbone=True)

✅ GradientReversal + PDLEncoderDANN definidos
   Params entrenables: 189,253 / 21,473,925


## SECCIÓN 5 — FUNCIONES DE PÉRDIDA

In [6]:
def ranking_loss(embeddings, pdl, cl_idx):
    """
    Ranking loss intra cell_line: para pares del mismo donante,
    penaliza si el embedding no respeta el orden de PDL.
    Margin ranking con margen proporcional a diferencia de PDL.
    """
    loss = torch.tensor(0.0, device=embeddings.device)
    n_pairs = 0

    # Calcular "score" como norma L2 del embedding (proxy de magnitud)
    scores = embeddings.norm(dim=1)

    # Iterar por cell_lines presentes en el batch
    unique_cl = cl_idx.unique()
    for cl in unique_cl:
        mask = cl_idx == cl
        if mask.sum() < 2:
            continue

        s = scores[mask]
        p = pdl[mask]

        # Todas las parejas
        n = s.shape[0]
        for i in range(n):
            for j in range(i + 1, n):
                if abs(p[i] - p[j]) < 0.05:  # skip si PDL muy similar
                    continue
                # i debería tener score menor si pdl[i] < pdl[j]
                if p[i] < p[j]:
                    margin = (p[j] - p[i]) * 0.5
                    loss += torch.relu(margin - (s[j] - s[i]))
                else:
                    margin = (p[i] - p[j]) * 0.5
                    loss += torch.relu(margin - (s[i] - s[j]))
                n_pairs += 1

    if n_pairs > 0:
        loss = loss / n_pairs
    return loss


def consistency_loss(emb1, emb2):
    """MSE entre embeddings de dos augmentations de la misma imagen."""
    return torch.mean((emb1 - emb2) ** 2)


print("✅ Ranking loss + consistency loss definidos")

✅ Ranking loss + consistency loss definidos


## SECCIÓN 6 — LOOP DE ENTRENAMIENTO GENÉRICO

In [7]:
def dann_alpha_schedule(epoch, max_epochs, max_alpha):
    """Schedule progresivo: empieza en 0, sube a max_alpha."""
    p = epoch / max_epochs
    return max_alpha * (2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0)


@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    all_pdl, all_hat, all_emb = [], [], []
    for img1, img2, pdl, cl_idx, sp_idx in loader:
        img1 = img1.to(device)
        pdl_hat, emb, _ = model(img1, dann_alpha=0.0)
        all_pdl.extend(pdl.numpy())
        all_hat.extend(pdl_hat.cpu().numpy())
        all_emb.append(emb.cpu().numpy())

    all_pdl = np.array(all_pdl)
    all_hat = np.array(all_hat)
    all_emb = np.vstack(all_emb)

    mae = np.mean(np.abs(all_pdl - all_hat))
    rho, p_val = spearmanr(all_pdl, all_hat)
    ss_res = np.sum((all_pdl - all_hat) ** 2)
    ss_tot = np.sum((all_pdl - all_pdl.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0

    return {
        "mae_norm": mae, "spearman": rho, "spearman_p": p_val, "r2": r2,
        "y_true": all_pdl, "y_pred": all_hat, "embeddings": all_emb,
    }


def train_ablation(df, ablation_id, use_ranking=False, use_consistency=False,
                   use_dann=False, verbose=True):
    """
    Entrena un modelo con las pérdidas especificadas, 3-fold CV.
    Retorna resultados por fold + embeddings.
    """
    criterion_pdl = nn.L1Loss()
    criterion_dann = nn.CrossEntropyLoss()
    folds = sorted(df[COL_FOLD].unique())

    results_per_fold = {}
    all_embeddings = []

    for val_fold in folds:
        if verbose:
            print(f"\n{'='*55}")
            print(f"  {ablation_id} | FOLD {val_fold}")
            print(f"{'='*55}")

        df_train = df[df[COL_FOLD] != val_fold]
        df_val = df[df[COL_FOLD] == val_fold]

        ds_train = AblationDataset(df_train, train_transform, train_transform_aug)
        ds_val = AblationDataset(df_val, val_transform, val_transform)
        dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=False)
        dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=0, pin_memory=False)

        # Baseline trivial
        trivial_mae = np.mean(np.abs(
            df_val[COL_PDL_NORM].values - df_train[COL_PDL_NORM].mean()))

        # Modelo
        model = PDLEncoderDANN(EMBEDDING_DIM, N_STUDY_PARTS,
                               freeze_backbone=True).to(DEVICE)
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=LR_HEAD, weight_decay=WEIGHT_DECAY)

        best_mae = float("inf")
        patience_counter = 0
        history = defaultdict(list)

        for epoch in range(EPOCHS):
            model.train()
            epoch_losses = defaultdict(float)
            n_batches = 0

            # DANN alpha schedule
            alpha = dann_alpha_schedule(epoch, EPOCHS, DANN_MAX) if use_dann else 0.0

            for img1, img2, pdl, cl_idx, sp_idx in dl_train:
                img1 = img1.to(DEVICE)
                img2 = img2.to(DEVICE)
                pdl = pdl.to(DEVICE)
                cl_idx = cl_idx.to(DEVICE)
                sp_idx = sp_idx.to(DEVICE)

                optimizer.zero_grad()

                # Forward pass 1
                pdl_hat, emb1, sp_logits = model(img1, dann_alpha=alpha)

                # --- L_pdl ---
                loss_pdl = criterion_pdl(pdl_hat, pdl)
                total_loss = LAMBDA_PDL * loss_pdl
                epoch_losses["pdl"] += loss_pdl.item()

                # --- L_rank ---
                if use_ranking:
                    loss_rank = ranking_loss(emb1, pdl, cl_idx)
                    total_loss += LAMBDA_RANK * loss_rank
                    epoch_losses["rank"] += loss_rank.item()

                # --- L_cons ---
                if use_consistency:
                    _, emb2, _ = model(img2, dann_alpha=0.0)
                    loss_cons = consistency_loss(emb1, emb2)
                    total_loss += LAMBDA_CONS * loss_cons
                    epoch_losses["cons"] += loss_cons.item()

                # --- L_dann ---
                if use_dann:
                    loss_dann = criterion_dann(sp_logits, sp_idx)
                    total_loss += LAMBDA_DANN * loss_dann
                    epoch_losses["dann"] += loss_dann.item()

                total_loss.backward()
                optimizer.step()
                n_batches += 1

            # Evaluate
            val_metrics = evaluate_model(model, dl_val, DEVICE)

            for k, v in epoch_losses.items():
                history[f"train_{k}"].append(v / n_batches)
            history["val_mae"].append(val_metrics["mae_norm"])
            history["val_spearman"].append(val_metrics["spearman"])

            improved = ""
            if val_metrics["mae_norm"] < best_mae:
                best_mae = val_metrics["mae_norm"]
                patience_counter = 0
                improved = "✓"
                torch.save(model.state_dict(),
                           os.path.join(OUTPUT_DIR, f"best_{ablation_id}_fold{val_fold}.pt"))
            else:
                patience_counter += 1

            if verbose and ((epoch + 1) % 10 == 0 or improved or epoch == 0):
                dann_str = f" | α={alpha:.3f}" if use_dann else ""
                print(f"  Ep {epoch+1:2d} | MAE: {val_metrics['mae_norm']:.4f} | "
                      f"ρ: {val_metrics['spearman']:.3f}{dann_str} {improved}")

            if patience_counter >= PATIENCE:
                if verbose:
                    print(f"  Early stopping ep {epoch+1}")
                break

        # Cargar mejor y evaluar
        model.load_state_dict(torch.load(
            os.path.join(OUTPUT_DIR, f"best_{ablation_id}_fold{val_fold}.pt"),
            weights_only=True))
        final = evaluate_model(model, dl_val, DEVICE)

        improvement = (1 - final["mae_norm"] / trivial_mae) * 100

        # Guardar embeddings con metadata
        df_emb = df_val.copy().reset_index(drop=True)
        df_emb["y_pred"] = final["y_pred"]
        for i in range(final["embeddings"].shape[1]):
            df_emb[f"emb_{i}"] = final["embeddings"][:, i]
        all_embeddings.append(df_emb)

        results_per_fold[val_fold] = {
            "n_val": len(df_val),
            "cell_lines_val": sorted(df_val[COL_CELL_LINE].unique()),
            "trivial_mae": trivial_mae,
            "model_mae": final["mae_norm"],
            "improvement_pct": improvement,
            "spearman": final["spearman"],
            "spearman_p": final["spearman_p"],
            "r2": final["r2"],
        }

        if verbose:
            print(f"  📊 MAE={final['mae_norm']:.4f} (mejora {improvement:+.1f}%) | "
                  f"ρ={final['spearman']:.3f} | R²={final['r2']:.3f}")

    df_embeddings = pd.concat(all_embeddings, ignore_index=True)
    return results_per_fold, df_embeddings


print("✅ Loop de entrenamiento definido")

✅ Loop de entrenamiento definido


## SECCIÓN 7 — CORRER ABLATIONS
Esto toma ~15-20 min en MPS para las 3 variantes.

In [8]:
ablation_configs = {
    "A1_rank": dict(use_ranking=True, use_consistency=False, use_dann=False),
    "A2_rank_cons": dict(use_ranking=True, use_consistency=True, use_dann=False),
    "A3_dann": dict(use_ranking=True, use_consistency=True, use_dann=True),
}

all_results = {}
all_embs = {}

for name, config in ablation_configs.items():
    print(f"\n{'#'*60}")
    print(f"  ABLATION: {name}")
    print(f"  Config: {config}")
    print(f"{'#'*60}")

    results, embs = train_ablation(df, name, **config)
    all_results[name] = results
    all_embs[name] = embs

print("\n✅ Todas las ablations completadas")


############################################################
  ABLATION: A1_rank
  Config: {'use_ranking': True, 'use_consistency': False, 'use_dann': False}
############################################################

  A1_rank | FOLD 0.0
   Params entrenables: 189,253 / 21,473,925
  Ep  1 | MAE: 0.1886 | ρ: 0.722 ✓
  Ep  2 | MAE: 0.1848 | ρ: 0.769 ✓
  Ep  3 | MAE: 0.1601 | ρ: 0.767 ✓
  Ep  4 | MAE: 0.1592 | ρ: 0.765 ✓
  Ep 10 | MAE: 0.1468 | ρ: 0.761 ✓
  Ep 20 | MAE: 0.1283 | ρ: 0.814 ✓
  Ep 30 | MAE: 0.1450 | ρ: 0.784 
  Early stopping ep 30


/var/folders/fy/k8jygkfn6hd63_wrlzhk82c40000gp/T/ipykernel_56930/4187597666.py:166: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_emb[f"emb_{i}"] = final["embeddings"][:, i]


  📊 MAE=0.1283 (mejora +44.7%) | ρ=0.814 | R²=0.564

  A1_rank | FOLD 1.0
   Params entrenables: 189,253 / 21,473,925
  Ep  1 | MAE: 0.1943 | ρ: 0.711 ✓
  Ep  3 | MAE: 0.1601 | ρ: 0.748 ✓
  Ep  4 | MAE: 0.1511 | ρ: 0.751 ✓
  Ep  5 | MAE: 0.1382 | ρ: 0.770 ✓
  Ep  6 | MAE: 0.1343 | ρ: 0.767 ✓
  Ep  8 | MAE: 0.1277 | ρ: 0.790 ✓
  Ep  9 | MAE: 0.1236 | ρ: 0.781 ✓
  Ep 10 | MAE: 0.1155 | ρ: 0.809 ✓
  Ep 20 | MAE: 0.1069 | ρ: 0.830 ✓
  Ep 30 | MAE: 0.1180 | ρ: 0.815 
  Early stopping ep 30
  📊 MAE=0.1069 (mejora +59.7%) | ρ=0.830 | R²=0.708

  A1_rank | FOLD 2.0
   Params entrenables: 189,253 / 21,473,925
  Ep  1 | MAE: 0.2462 | ρ: 0.397 ✓
  Ep  2 | MAE: 0.2347 | ρ: 0.433 ✓
  Ep  3 | MAE: 0.2195 | ρ: 0.484 ✓
  Ep  4 | MAE: 0.2151 | ρ: 0.580 ✓
  Ep  6 | MAE: 0.1996 | ρ: 0.597 ✓
  Ep  7 | MAE: 0.1899 | ρ: 0.612 ✓
  Ep 10 | MAE: 0.1872 | ρ: 0.617 ✓
  Ep 20 | MAE: 0.2044 | ρ: 0.558 
  Early stopping ep 20
  📊 MAE=0.1872 (mejora +27.9%) | ρ=0.617 | R²=0.406

#####################################

## SECCIÓN 8 — TABLA COMPARATIVA

In [9]:
print("\n" + "=" * 80)
print("  TABLA COMPARATIVA — ABLATIONS")
print("=" * 80)

# Incluir A0 (baseline) de referencia manual si quieres, o solo A1-A3
rows = []
for name, results in all_results.items():
    maes = [r["model_mae"] for r in results.values()]
    spears = [r["spearman"] for r in results.values()]
    imps = [r["improvement_pct"] for r in results.values()]
    worst_spear = min(spears)
    rows.append({
        "Ablation": name,
        "MAE_mean": f"{np.mean(maes):.4f}",
        "MAE_std": f"{np.std(maes):.4f}",
        "Spearman_mean": f"{np.mean(spears):.3f}",
        "Spearman_std": f"{np.std(spears):.3f}",
        "Spearman_worst": f"{worst_spear:.3f}",
        "Mejora_mean": f"{np.mean(imps):+.1f}%",
        "Fold2_spearman": f"{results[sorted(results.keys())[-1]]['spearman']:.3f}",
    })

df_comp = pd.DataFrame(rows)
print(df_comp.to_string(index=False))

# Chequeo PIDA
print(f"\n📋 Criterios PIDA:")
for name in all_results:
    spears = [r["spearman"] for r in all_results[name].values()]
    imps = [r["improvement_pct"] for r in all_results[name].values()]
    s_ok = "✓" if np.mean(spears) >= 0.6 else "✗"
    m_ok = "✓" if np.mean(imps) >= 25 else "✗"
    w_ok = "✓" if min(spears) >= 0.6 else "✗"
    print(f"  {name:15s} | ρ≥0.6: {s_ok} ({np.mean(spears):.3f}) | "
          f"mejora≥25%: {m_ok} ({np.mean(imps):+.1f}%) | "
          f"worst fold≥0.6: {w_ok} ({min(spears):.3f})")


  TABLA COMPARATIVA — ABLATIONS
    Ablation MAE_mean MAE_std Spearman_mean Spearman_std Spearman_worst Mejora_mean Fold2_spearman
     A1_rank   0.1408  0.0340         0.754        0.097          0.617      +44.1%          0.617
A2_rank_cons   0.1446  0.0283         0.736        0.084          0.617      +42.6%          0.617
     A3_dann   0.1447  0.0327         0.734        0.100          0.595      +42.5%          0.595

📋 Criterios PIDA:
  A1_rank         | ρ≥0.6: ✓ (0.754) | mejora≥25%: ✓ (+44.1%) | worst fold≥0.6: ✓ (0.617)
  A2_rank_cons    | ρ≥0.6: ✓ (0.736) | mejora≥25%: ✓ (+42.6%) | worst fold≥0.6: ✓ (0.617)
  A3_dann         | ρ≥0.6: ✓ (0.734) | mejora≥25%: ✓ (+42.5%) | worst fold≥0.6: ✗ (0.595)


## SECCIÓN 9 — BATCH PROBE INCREMENTAL
Mide si z agrega información de batch *más allá* de lo que ya predicen
PDL + cell_line (variables estructurales).

In [10]:
print("\n" + "=" * 80)
print("  BATCH PROBE INCREMENTAL — ¿z agrega batch extra?")
print("=" * 80)

probe_results = {}

for name, df_emb in all_embs.items():
    print(f"\n  --- {name} ---")

    emb_cols = [c for c in df_emb.columns if c.startswith("emb_")]
    X_emb = df_emb[emb_cols].values

    # Target: study_part
    y_sp = df_emb[COL_STUDY_PART].values
    le_sp = LabelEncoder()
    y_enc = le_sp.fit_transform(y_sp)
    n_classes = len(np.unique(y_enc))

    if n_classes < 2:
        print("    Solo 1 clase de study_part — skip")
        continue

    # --- Modelo A: solo PDL + cell_line → study_part ---
    X_struct = pd.get_dummies(df_emb[[COL_PDL_NORM, COL_CELL_LINE]],
                              columns=[COL_CELL_LINE]).values.astype(float)
    clf_a = LogisticRegression(max_iter=2000, random_state=SEED, C=1.0)
    clf_a.fit(X_struct, y_enc)
    prob_a = clf_a.predict_proba(X_struct)
    auc_a = roc_auc_score(y_enc, prob_a, multi_class="ovr", average="macro")

    # --- Modelo B: PDL + cell_line + z → study_part ---
    X_full = np.hstack([X_struct, X_emb])
    clf_b = LogisticRegression(max_iter=2000, random_state=SEED, C=1.0)
    clf_b.fit(X_full, y_enc)
    prob_b = clf_b.predict_proba(X_full)
    auc_b = roc_auc_score(y_enc, prob_b, multi_class="ovr", average="macro")

    # --- Modelo C: solo z → study_part (raw batch probe) ---
    clf_c = LogisticRegression(max_iter=2000, random_state=SEED, C=1.0)
    clf_c.fit(X_emb, y_enc)
    prob_c = clf_c.predict_proba(X_emb)
    auc_c = roc_auc_score(y_enc, prob_c, multi_class="ovr", average="macro")

    # --- PDL bins (señal biológica, debería mantenerse) ---
    pdl_bins = pd.cut(df_emb[COL_PDL_NORM], bins=3, labels=[0, 1, 2])
    clf_pdl = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
    clf_pdl.fit(X_emb, pdl_bins)
    prob_pdl = clf_pdl.predict_proba(X_emb)
    auc_pdl = roc_auc_score(pdl_bins, prob_pdl, multi_class="ovr", average="macro")

    delta = auc_b - auc_a
    status_delta = "✅ OK" if delta <= 0.10 else "⚠️ z agrega batch"
    status_raw = "✅" if auc_c < 0.75 else "⚠️ CONTAMINADO"
    status_pdl = "✅" if auc_pdl >= 0.85 else "⚠️ perdió señal"

    print(f"    study_part AUC (raw z):           {auc_c:.3f} {status_raw}")
    print(f"    study_part AUC (PDL+cell_line):   {auc_a:.3f}")
    print(f"    study_part AUC (PDL+cell_line+z): {auc_b:.3f}")
    print(f"    DELTA incremental:                {delta:+.3f} {status_delta}")
    print(f"    PDL bins AUC:                     {auc_pdl:.3f} {status_pdl}")

    probe_results[name] = {
        "raw_z_auc": auc_c, "struct_auc": auc_a, "full_auc": auc_b,
        "delta": delta, "pdl_bins_auc": auc_pdl,
    }

# Guardar
with open(os.path.join(OUTPUT_DIR, "batch_probe_incremental.json"), "w") as f:
    json.dump(probe_results, f, indent=2)
print(f"\n💾 Guardado: batch_probe_incremental.json")


  BATCH PROBE INCREMENTAL — ¿z agrega batch extra?

  --- A1_rank ---
    study_part AUC (raw z):           0.996 ⚠️ CONTAMINADO
    study_part AUC (PDL+cell_line):   0.850
    study_part AUC (PDL+cell_line+z): 0.998
    DELTA incremental:                +0.148 ⚠️ z agrega batch
    PDL bins AUC:                     0.963 ✅

  --- A2_rank_cons ---
    study_part AUC (raw z):           0.973 ⚠️ CONTAMINADO
    study_part AUC (PDL+cell_line):   0.850
    study_part AUC (PDL+cell_line+z): 0.982
    DELTA incremental:                +0.132 ⚠️ z agrega batch
    PDL bins AUC:                     0.904 ✅

  --- A3_dann ---
    study_part AUC (raw z):           0.952 ⚠️ CONTAMINADO
    study_part AUC (PDL+cell_line):   0.850
    study_part AUC (PDL+cell_line+z): 0.975
    DELTA incremental:                +0.125 ⚠️ z agrega batch
    PDL bins AUC:                     0.910 ✅

💾 Guardado: batch_probe_incremental.json


## SECCIÓN 10 — FUSION-READINESS (correlación parcial)
Correlación de z_img con telómero/mtDNA **controlando** por PDL y cell_line.

In [11]:
print("\n" + "=" * 80)
print("  FUSION-READINESS — Correlación parcial con biomarcadores")
print("=" * 80)

def partial_correlation(x, y, covariates):
    """
    Correlación parcial de Spearman entre x e y,
    controlando por covariates (matrix n×k).
    Método: residualizar x e y contra covariates, luego correlacionar residuos.
    """
    from sklearn.linear_model import LinearRegression

    reg_x = LinearRegression().fit(covariates, x)
    res_x = x - reg_x.predict(covariates)

    reg_y = LinearRegression().fit(covariates, y)
    res_y = y - reg_y.predict(covariates)

    return spearmanr(res_x, res_y)


fusion_results = {}

for name, df_emb in all_embs.items():
    print(f"\n  --- {name} ---")

    emb_cols = [c for c in df_emb.columns if c.startswith("emb_")]

    # PCA del embedding
    from sklearn.decomposition import PCA
    pca = PCA(n_components=1, random_state=SEED)
    df_emb["emb_pc1"] = pca.fit_transform(df_emb[emb_cols].values)[:, 0]

    # Agregar por sample_id
    bio_cols = [COL_TELOMERE, COL_MTDNA]
    agg_cols = {"emb_pc1": "mean", COL_PDL_NORM: "first", COL_CELL_LINE: "first"}
    for col in bio_cols:
        if col in df_emb.columns:
            agg_cols[col] = "first"

    agg = df_emb.dropna(subset=[c for c in bio_cols if c in df_emb.columns])
    if len(agg) == 0:
        print("    No hay datos de biomarcadores — skip")
        continue

    agg = agg.groupby(COL_SAMPLE_ID).agg(agg_cols).reset_index()

    # Covariates: PDL + cell_line one-hot
    cov = pd.get_dummies(agg[[COL_PDL_NORM, COL_CELL_LINE]],
                         columns=[COL_CELL_LINE]).values.astype(float)

    fusion_results[name] = {}

    for col in bio_cols:
        if col not in agg.columns:
            continue
        mask = agg[col].notna()
        n_valid = mask.sum()
        if n_valid < 15:
            print(f"    {col}: solo {n_valid} muestras — skip")
            continue

        x = agg.loc[mask, "emb_pc1"].values
        y = agg.loc[mask, col].values
        c = cov[mask.values]

        # Correlación simple
        rho_simple, p_simple = spearmanr(x, y)

        # Correlación parcial (controlando PDL + cell_line)
        rho_partial, p_partial = partial_correlation(x, y, c)

        sig_s = "***" if p_simple < 0.001 else "**" if p_simple < 0.01 else "*" if p_simple < 0.05 else "ns"
        sig_p = "***" if p_partial < 0.001 else "**" if p_partial < 0.01 else "*" if p_partial < 0.05 else "ns"

        print(f"    {col:25s} | simple ρ={rho_simple:+.3f} {sig_s} | "
              f"parcial ρ={rho_partial:+.3f} {sig_p} | n={n_valid}")

        fusion_results[name][col] = {
            "simple_rho": rho_simple, "simple_p": p_simple,
            "partial_rho": rho_partial, "partial_p": p_partial,
            "n": int(n_valid),
        }

with open(os.path.join(OUTPUT_DIR, "fusion_readiness_partial.json"), "w") as f:
    json.dump(fusion_results, f, indent=2, default=str)
print(f"\n💾 Guardado: fusion_readiness_partial.json")


  FUSION-READINESS — Correlación parcial con biomarcadores

  --- A1_rank ---
    telomere_length           | simple ρ=+0.181 ns | parcial ρ=-0.089 ns | n=77
    mtdna_cn                  | simple ρ=-0.045 ns | parcial ρ=+0.154 ns | n=77

  --- A2_rank_cons ---
    telomere_length           | simple ρ=+0.218 ns | parcial ρ=+0.007 ns | n=77
    mtdna_cn                  | simple ρ=+0.004 ns | parcial ρ=+0.252 * | n=77

  --- A3_dann ---
    telomere_length           | simple ρ=+0.163 ns | parcial ρ=-0.029 ns | n=77
    mtdna_cn                  | simple ρ=-0.017 ns | parcial ρ=+0.193 ns | n=77

💾 Guardado: fusion_readiness_partial.json


/var/folders/fy/k8jygkfn6hd63_wrlzhk82c40000gp/T/ipykernel_56930/723641367.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_emb["emb_pc1"] = pca.fit_transform(df_emb[emb_cols].values)[:, 0]
/var/folders/fy/k8jygkfn6hd63_wrlzhk82c40000gp/T/ipykernel_56930/723641367.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_emb["emb_pc1"] = pca.fit_transform(df_emb[emb_cols].values)[:, 0]
/var/folders/fy/k8jygkfn6hd63_wrlzhk82c40000gp/T/ipykernel_56930/723641367.py:32: PerformanceWarning: DataFrame is highly fragmented.  This 

## SECCIÓN 11 — REPORTE FINAL

In [12]:
print("\n" + "=" * 80)
print("  REPORTE FINAL — ABLATION COMPLETA")
print("=" * 80)

# Tabla de comparación completa
print("\n📊 Rendimiento por ablation:")
for name in all_results:
    r = all_results[name]
    spears = [v["spearman"] for v in r.values()]
    maes = [v["model_mae"] for v in r.values()]
    imps = [v["improvement_pct"] for v in r.values()]
    print(f"  {name:15s} | ρ={np.mean(spears):.3f}±{np.std(spears):.3f} | "
          f"MAE={np.mean(maes):.4f} | mejora={np.mean(imps):+.1f}% | "
          f"worst ρ={min(spears):.3f}")

print(f"\n📊 Batch probe incremental (study_part):")
for name, pr in probe_results.items():
    print(f"  {name:15s} | raw={pr['raw_z_auc']:.3f} | "
          f"struct={pr['struct_auc']:.3f} | +z={pr['full_auc']:.3f} | "
          f"Δ={pr['delta']:+.3f} | PDL bins={pr['pdl_bins_auc']:.3f}")

print(f"\n📊 Fusion-readiness (correlación parcial, controlando PDL+cell_line):")
for name, fr in fusion_results.items():
    for col, vals in fr.items():
        print(f"  {name:15s} | {col:25s} | ρ_parcial={vals['partial_rho']:+.3f} "
              f"(p={vals['partial_p']:.2e}) | n={vals['n']}")

# Guardar reporte completo
report = {
    "timestamp": datetime.now().isoformat(),
    "config": {
        "manifest": os.path.basename(MANIFEST_PATH),
        "filter": "10x only",
        "lambdas": {"pdl": LAMBDA_PDL, "rank": LAMBDA_RANK,
                    "cons": LAMBDA_CONS, "dann": LAMBDA_DANN, "dann_max": DANN_MAX},
    },
    "results": {name: {str(k): {kk: vv for kk, vv in v.items()
                                 if kk not in ["y_true", "y_pred", "embeddings"]}
                       for k, v in r.items()}
                for name, r in all_results.items()},
    "batch_probe": probe_results,
    "fusion_readiness": fusion_results,
}

report_path = os.path.join(OUTPUT_DIR, "ablation_report.json")
with open(report_path, "w") as f:
    json.dump(report, f, indent=2, default=str)

print(f"\n💾 Reporte: {report_path}")
print(f"\n{'='*60}")
print(f"  ABLATION COMPLETA")
print(f"{'='*60}")


  REPORTE FINAL — ABLATION COMPLETA

📊 Rendimiento por ablation:
  A1_rank         | ρ=0.754±0.097 | MAE=0.1408 | mejora=+44.1% | worst ρ=0.617
  A2_rank_cons    | ρ=0.736±0.084 | MAE=0.1446 | mejora=+42.6% | worst ρ=0.617
  A3_dann         | ρ=0.734±0.100 | MAE=0.1447 | mejora=+42.5% | worst ρ=0.595

📊 Batch probe incremental (study_part):
  A1_rank         | raw=0.996 | struct=0.850 | +z=0.998 | Δ=+0.148 | PDL bins=0.963
  A2_rank_cons    | raw=0.973 | struct=0.850 | +z=0.982 | Δ=+0.132 | PDL bins=0.904
  A3_dann         | raw=0.952 | struct=0.850 | +z=0.975 | Δ=+0.125 | PDL bins=0.910

📊 Fusion-readiness (correlación parcial, controlando PDL+cell_line):
  A1_rank         | telomere_length           | ρ_parcial=-0.089 (p=4.43e-01) | n=77
  A1_rank         | mtdna_cn                  | ρ_parcial=+0.154 (p=1.81e-01) | n=77
  A2_rank_cons    | telomere_length           | ρ_parcial=+0.007 (p=9.51e-01) | n=77
  A2_rank_cons    | mtdna_cn                  | ρ_parcial=+0.252 (p=2.73e-02) |

## SECCIÓN 12 — INTERPRETACIÓN

### Escenarios posibles y siguiente paso:

| A3 batch Δ | A3 Spearman worst | Fusion parcial | Interpretación | Acción |
|-----------|-------------------|----------------|----------------|--------|
| Δ ≤ 0.10 | ≥ 0.60 | ρ significativa | ✅ z limpio, señal real | Proceder a MVP-2 |
| Δ ≤ 0.10 | ≥ 0.60 | ρ no significativa | 📊 z limpio, n bajo | Aceptar, fusión mejorará con ómicas |
| Δ > 0.10 | ≥ 0.60 | cualquiera | ⚠️ Batch residual, señal parcial | Subir DANN_MAX, más epochs |
| Δ ≤ 0.10 | < 0.60 | cualquiera | ⚠️ DANN mató señal | Bajar DANN_MAX, revisar schedule |
| Δ > 0.10 | < 0.60 | cualquiera | ❌ Señal era batch | Fundamentalmente replantear |